# XGBoost Regression - California Housing Dataset

## Step 1: Import Required Libraries

We need to import essential libraries for data manipulation, visualization, and machine learning:
- **numpy**: For numerical operations
- **pandas**: For data manipulation and analysis
- **matplotlib**: For plotting and visualization
- **sklearn**: For machine learning algorithms and datasets
- **xgboost**: For XGBoost algorithm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

## Step 2: Load the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["Price"] = housing.target

df.head()

## Step 3: Data Exploration

Understanding the dataset structure and statistics is crucial before building any model.

In [ ]:
df.shape
df.info()
df.describe()

## Step 4: Check for Missing Values

In [ ]:
df.isnull().sum()

## Step 5: Check for Duplicate Rows

In [ ]:
df.duplicated().sum()

## Step 6: Split Features and Target

In [ ]:
X = df.drop("Price",axis=1)

y = df["Price"]

## Step 7: Data Visualization - Histograms

In [ ]:
df.hist(
    figsize=(14,10),
    bins=30
)

plt.show()

## Step 8: Correlation Analysis

In [ ]:
corr=df.corr()

corr["Price"].sort_values(
    ascending=False
)
import seaborn as sns

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.show()

## Step 9: Outlier Detection - Boxplots

In [ ]:
for col in df.columns:

    plt.figure()

    plt.boxplot(df[col])

    plt.title(col)

    plt.show()

## Step 10: Feature Scaling (Optional for XGBoost)

**Note:** XGBoost is relatively insensitive to feature scaling, but we'll scale for consistency with other models.

**StandardScaler Formula:**
$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

X_scaled=pd.DataFrame(
    X_scaled,
    columns=X.columns
)

## Step 11: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=\
train_test_split(
X_scaled,
y,
test_size=0.2,
random_state=42
)
X_train.shape
X_test.shape

## XGBoost Regression Formula and Concepts

**XGBoost (Extreme Gradient Boosting) Concept:**

XGBoost is an optimized distributed gradient boosting library designed to be highly efficient, flexible, and portable. It implements machine learning algorithms under the Gradient Boosting framework.

**Key Improvements over Traditional Gradient Boosting:**

1. **Regularization**: L1 and L2 regularization to prevent overfitting
2. **Parallel Processing**: Can parallelize tree construction
3. **Tree Pruning**: Uses max depth approach rather than pre-pruning
4. **Handling Missing Values**: Built-in handling of missing data
5. **Cross-validation**: Built-in cross-validation
6. **Custom Loss Functions**: Can define custom objective functions

**Mathematical Formulation:**

The objective function consists of two parts:

$$Obj(\Theta) = L(\Theta) + \Omega(\Theta)$$

Where:
- **L(Θ)** = Training loss (e.g., MSE, MAE)
- **Ω(Θ)** = Regularization term

**Regularization Term:**

$$\Omega(f) = \gamma T + \frac{1}{2}\lambda ||w||^2$$

Where:
- **γ (gamma)** = Minimum loss reduction required to make a split
- **T** = Number of leaves in the tree
- **λ (lambda)** = L2 regularization term on leaf weights
- **w** = Leaf weights

**Key Hyperparameters:**
- **n_estimators**: Number of boosting rounds (trees)
- **learning_rate (eta)**: Step size shrinkage (0.01-0.3)
- **max_depth**: Maximum depth of each tree (typically 3-10)
- **min_child_weight**: Minimum sum of instance weight in a child
- **subsample**: Fraction of samples to use for each tree (0.5-1.0)
- **colsample_bytree**: Fraction of features to use for each tree (0.5-1.0)
- **gamma**: Minimum loss reduction for split (0-10)
- **lambda**: L2 regularization term (0-10)
- **alpha**: L1 regularization term (0-10)

**Advantages:**
- **High performance**: Often achieves state-of-the-art results
- **Fast execution**: Optimized algorithm with parallel processing
- **Regularization**: Built-in L1 and L2 regularization
- **Handles missing values**: No need for imputation
- **Flexible**: Can optimize custom loss functions
- **Cross-validation**: Built-in CV functionality
- **Feature importance**: Provides detailed feature importance

**Disadvantages:**
- **Complex**: Many hyperparameters to tune
- **Overfitting**: Can overfit if not properly regularized
- **Memory intensive**: Can use significant memory
- **Less interpretable**: Harder to understand than simpler models
- **Requires installation**: Not part of standard sklearn

**Key Concepts:**
- **Gradient Boosting**: Sequential ensemble method
- **Regularization**: L1 and L2 penalties prevent overfitting
- **Tree Pruning**: Post-pruning for better generalization
- **Parallel Processing**: Faster training than traditional GB

In [ ]:
try:
    import xgboost as xgb
    from sklearn.metrics import (
        mean_absolute_error,
        mean_squared_error,
        r2_score
    )
    print("XGBoost imported successfully!")
except ImportError:
    print("XGBoost not installed. Install with: pip install xgboost")

## Train XGBoost with Default Parameters

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = xgb_model.predict(
    X_test
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse**0.5

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R²:",r2)

## Model Information

In [ ]:
print("Number of Estimators:", xgb_model.n_estimators)
print("Learning Rate:", xgb_model.learning_rate)
print("Max Depth:", xgb_model.max_depth)
print("Number of Features:", xgb_model.n_features_in_)

## Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("XGBoost Feature Importance")
plt.tight_layout()
plt.show()

## Visualization: Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()]
)

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.title(
    "XGBoost Regression"
)

plt.show()

## Residual Plot

In [ ]:
residuals = y_test-y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals
)

plt.axhline(y=0)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Residuals"
)

plt.title(
    "Residual Plot"
)

plt.show()

## Hyperparameter Tuning: Number of Estimators

Let's test different numbers of estimators to see how it affects performance.

In [ ]:
n_estimators_list = [50, 100, 200, 500]

results = []

for n in n_estimators_list:
    model = xgb.XGBRegressor(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    
    results.append({
        'n_estimators': n,
        'R²': r2
    })

results_df = pd.DataFrame(results)
print(results_df)

## Hyperparameter Tuning: Learning Rate

The learning rate controls how much each tree contributes.

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.2]

results = []

for lr in learning_rates:
    model = xgb.XGBRegressor(n_estimators=100, learning_rate=lr, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    
    results.append({
        'learning_rate': lr,
        'R²': r2
    })

results_df = pd.DataFrame(results)
print(results_df)

## Compare with Gradient Boosting

Let's compare XGBoost with sklearn's Gradient Boosting to see the performance difference.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Gradient Boosting (sklearn)
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
r2_gb = r2_score(y_test, y_pred_gb)

# XGBoost
xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
r2_xgb = r2_score(y_test, y_pred_xgb)

print("Comparison Results:")
print(f"Gradient Boosting (sklearn): R² = {r2_gb:.4f}")
print(f"XGBoost: R² = {r2_xgb:.4f}")
print(f"Difference: {(r2_xgb - r2_gb):.4f}")

## Summary

XGBoost Regression provides:
- **High performance**: Often achieves state-of-the-art results
- **Fast execution**: Optimized algorithm with parallel processing
- **Regularization**: Built-in L1 and L2 regularization
- **Handles missing values**: No need for imputation
- **Feature importance**: Detailed feature importance analysis
- **Cross-validation**: Built-in CV functionality

**Key advantages over sklearn Gradient Boosting:**
- **Faster training**: Optimized implementation
- **Better regularization**: More sophisticated regularization
- **Parallel processing**: Can utilize multiple cores
- **Tree pruning**: Better generalization through post-pruning
- **Missing value handling**: Built-in support

**Key considerations:**
- **Complex hyperparameters**: Many parameters to tune
- **Overfitting risk**: Requires careful regularization
- **Memory usage**: Can be memory-intensive
- **Installation required**: Not part of standard sklearn

**Best practices:**
- Use cross-validation for hyperparameter tuning
- Start with default parameters and tune gradually
- Use smaller learning rates with more estimators
- Monitor training vs validation performance
- Use early stopping to prevent overfitting
- Consider using DMatrix for large datasets